In [18]:
import pandas as pd
import numpy as np

df  = pd.read_excel("basedados.xlsx")

dataframe_dados_clientes = df.iloc[: , 1:19] #pega todas as linhas da coluna da 1 a 18
dataframe_gabarito = df.iloc[: , 19] #pega todas as linhas da coluna 20 (gabarito)

array_dados_clientes = dataframe_dados_clientes.values
array_gabarito = dataframe_gabarito.values


In [19]:
def criar_cromossomos(qtd_cromossomos: int=6, qtd_genes: int = 19) -> np.ndarray:
    """
    Cria uma matriz onde cada linha e um cromossomo e cada coluna e um gene
    """
    # Passamos as duas dimensões para o rand: linhas (indivíduos) e colunas (genes)
    array_cromossomos = -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)
    
    return array_cromossomos

In [20]:
cromossomos = criar_cromossomos()

In [21]:
print(cromossomos)

[[ 0.75678365 -0.07984997 -0.09675771 -0.79175433 -0.12826131 -0.30763699
  -0.21431984 -0.45534714 -0.06963958  0.21677848 -0.70565468 -0.00698956
   0.66092682  0.86571583  0.92323761 -0.62107332  0.19709876 -0.96842206
   0.96739832]
 [ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
  -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
  -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
  -0.04361909]
 [-0.33852982 -0.06224669 -0.30531546  0.92424173 -0.71705082 -0.23045857
   0.58501681  0.94871067 -0.2874281   0.72527371 -0.96368506  0.64323855
   0.35319031 -0.55804994 -0.65136115  0.16075666  0.57416479  0.1546739
   0.63634022]
 [ 0.23523421 -0.91010665 -0.27998229  0.53865249 -0.58598641  0.96333224
  -0.25029987 -0.74461256  0.6347373   0.67372972  0.88846973 -0.75360712
   0.00911487  0.93683948 -0.92708533  0.56378028  0.01831356 -0.16142216
  -0.15684531]
 [-0.60999872 -0.10199822  0.16737114 -0.57514993 -0.3

In [22]:
print(cromossomos[0,0])

0.7567836465775586


In [23]:
def calcular_fitness(cromossomos: np.ndarray, array_dados_clientes: np.ndarray, array_gabarito: np.ndarray) -> np.ndarray:
    """
    Pega cada linha do array_cromossomos (menos o primeiro termo) e itera sobre cada linha da array de dados clientes
    cada linha (iteracao) e um produto escalar
    cada iteracao vai gerar um vetor coluna contendo 0 e 1 chamado de vetor hipotese

    depois comparar o vetor hipotese de cada cromossomo com o vetor gabarito e calcular a porcentagem de acerto

    depois calcular o fitness seguindo a formula 

    percentual_adimplente = quantidade de 1 no vetor hipotese/ total de 1 no gabarito
    percentual_inadimplente - quantidade de 0 no vetor hipotese/ total de 0 no gabarito

    fitness = percentual_adimplente * percentual_inadimplente

    cada cromossomo vai ter um fitness. Somar todos os fitness e calcular a porcentagem relativa de acerta de cada cromossomo
    com base nisso
    """

    total_adimplentes = np.sum(array_gabarito == 1)
    total_inadimplentes = np.sum(array_gabarito == 0)
    
    lista_hipotese = []

    #iteracao pra cada cromossomo
    for linha in cromossomos:  # o for numa array vai de linha em linha automaticamente
        
        bias = linha[0]
        genes = linha[1:]

        q = np.dot(array_dados_clientes, genes) + bias

        #claudio ajudou, onde cada elemento de Q for maior igual a zero troque por 1, se nao troque por zero
        vetor_hipotese = np.where(q >= 0, 1, 0)

        acertos_adimplentes = np.sum((vetor_hipotese == 1) & (array_gabarito == 1)) #se a hipotese for 1 e o gabarito for 1 ele acertou, entao contabiliza
        acertos_inadimplentes = np.sum((vetor_hipotese == 0) & (array_gabarito == 0)) #se a hipotese for 0 e o gabarito for 0 ele acertou, entao contabiliza
        # se nao o & da false e ele nao soma 


        percentual_adimplente = acertos_adimplentes / total_adimplentes
        percentual_inadimplente = acertos_inadimplentes / total_inadimplentes

        fitness = percentual_adimplente * percentual_inadimplente
        
        lista_hipotese.append(fitness)

    
    return np.array(lista_hipotese) #array com o fitness de cada cromossomo

In [24]:
vetor_fitnesses = calcular_fitness(cromossomos, array_dados_clientes, array_gabarito)
print(vetor_fitnesses)

[0.14254386 0.45614035 0.01535088 0.21052632 0.         0.12280702]


In [25]:
def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
        """
        Recebe um array de fitnesses e retorna um array de fitnesses percentualizados.
        """
        soma_fitness = np.sum(vetor_fitnesses)
        percentual_fitness = vetor_fitnesses / soma_fitness
        return percentual_fitness

In [26]:
percentual_fitnesses = fitness_percentual(vetor_fitnesses)
print(percentual_fitnesses)

[0.15046296 0.48148148 0.0162037  0.22222222 0.         0.12962963]


In [27]:
def selecionar_pais_roleta(cromossomos: np.ndarray, percentual_fitnesses: np.ndarray):
    """
    gera dois numeros aleatorios entre 0 e 1 pra escolher quem serao os pais
    """
    # 1. Cria as fronteiras da roleta usando a soma acumulada
    roleta_acumulada = np.cumsum(percentual_fitnesses)
    
    # 2. Gera dois números aleatórios independentes entre 0 e 1
    random_pai = np.random.rand()
    random_mae = np.random.rand()
    
    # 3. Descobre matematicamente em qual fatia os números caíram
    indice_pai = np.searchsorted(roleta_acumulada, random_pai)
    indice_mae = np.searchsorted(roleta_acumulada, random_mae)
    
    # 4. Extrai as linhas correspondentes da sua matriz de população
    pai = cromossomos[indice_pai]
    mae = cromossomos[indice_mae]
    
    return pai, mae

In [28]:
pai, mae = selecionar_pais_roleta(cromossomos, percentual_fitnesses)

In [29]:
print(pai)
print(mae)

[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]
[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]


In [30]:
def crossover(pai: np.ndarray, mae: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Gera 3 filhos com ponto de corte independente cada um.
    Todos seguem o padrão: [ pai[:corte] | mae[corte:] ]
    """
    c1 = np.random.randint(1, len(pai))
    c2 = np.random.randint(1, len(pai))
    c3 = np.random.randint(1, len(pai))

    filho1 = np.concatenate([pai[:c1], mae[c1:]])
    filho2 = np.concatenate([pai[:c2], mae[c2:]])
    filho3 = np.concatenate([pai[:c3], mae[c3:]])

    return filho1, filho2, filho3

In [31]:
filho1, filho2, filho3 = crossover(pai, mae)
print(filho1)
print(filho2)
print(filho3)

[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]
[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]
[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]


In [32]:
def mutar(filho1: np.ndarray, filho2: np.ndarray, filho3: np.ndarray,) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Seleciona um gene aleatório em cada filho e substitui
    por um novo valor sorteado em [-1, 1].
    """
    for filho in [filho1, filho2, filho3]:
        indice = np.random.randint(0, len(filho))
        filho[indice] = -1 + 2 * np.random.rand()

    return filho1, filho2, filho3

In [33]:
filho1, filho2, filho3 = mutar(filho1, filho2, filho3)
print(filho1)
print(filho2)
print(filho3)

[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218 -0.51326625 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]
[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.31523059  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868  0.80522397  0.9719279
 -0.04361909]
[ 0.11106644  0.32685438 -0.15118256  0.16075037 -0.03294625  0.98750926
 -0.15633426 -0.98851085  0.23062218  0.22398602 -0.00554632  0.29914206
 -0.77373514 -0.40422745 -0.74256819  0.47279868 -0.56495336  0.9719279
 -0.04361909]


In [34]:
def atualizar_populacao(cromossomos: np.ndarray,vetor_fitnesses: np.ndarray,filho1: np.ndarray,filho2: np.ndarray,filho3: np.ndarray,array_dados_clientes: np.ndarray,array_gabarito: np.ndarray,) -> np.ndarray:
    """
    Calcula o fitness dos 3 filhos, seleciona os 2 melhores,
    e substitui os 2 piores cromossomos da população por eles.
    """
    # fitness dos 3 filhos
    filhos = np.array([filho1, filho2, filho3])
    fitnesses_filhos = calcular_fitness(filhos, array_dados_clientes, array_gabarito)

    # 2 melhores filhos (maiores fitnesses)
    indices_melhores_filhos = np.argsort(fitnesses_filhos)[-2:]

    # 2 piores cromossomos da população (menores fitnesses)
    indices_piores = np.argsort(vetor_fitnesses)[:2]

    # substitui
    nova_populacao = cromossomos.copy()
    for i, idx_pior in enumerate(indices_piores):
        nova_populacao[idx_pior] = filhos[indices_melhores_filhos[i]]

    return nova_populacao